# Image-Based Waste Classification with ResNet50 (TrashNet)

End-to-end notebook implementing the full methodology:
1. Dataset download + stratified 70/15/15 split
2. ResNet50 + staged/progressive unfreezing (baseline run)
3. Optuna hyperparameter search (Section 4.2)
4. Data augmentation ablation (Section 4.3)
5. Final evaluation on held-out test split (Section 4.4)

Run top to bottom on a Colab GPU runtime (Runtime > Change runtime type > T4 GPU).

## 0. Setup

In [ ]:
!pip install -q optuna scikit-learn pandas numpy matplotlib pillow tqdm

### Get the dataset

Upload your `kaggle.json` API token (Kaggle > Account > Create New Token), then run the cell below.
This downloads and unzips TrashNet into `data/raw/` with one folder per class.

In [ ]:
import os
from pathlib import Path

# Uncomment if running in Colab and you need to upload kaggle.json:
# from google.colab import files
# files.upload()  # select kaggle.json

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
if Path("kaggle.json").exists():
    os.system("cp kaggle.json ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json")

os.makedirs("data", exist_ok=True)
os.system("pip install -q kaggle")
os.system("kaggle datasets download -d feyzazkefe/trashnet -p data/ --unzip")

# The Kaggle mirror sometimes nests one extra folder level; normalize to data/raw/
import shutil
raw = Path("data/raw")
if not raw.exists():
    # find the folder that actually contains the 6 class subfolders
    candidates = [p for p in Path("data").rglob("*") if p.is_dir() and
                  {"glass", "paper", "cardboard", "plastic", "metal", "trash"}.issubset(
                      {c.name for c in p.iterdir() if c.is_dir()})]
    if candidates:
        shutil.move(str(candidates[0]), str(raw))
print("Raw data ready at:", raw.resolve())

### Write the project source files

Instead of duplicating logic across notebook cells, we write out the same
`src/` modules used in the standalone script version and import them directly.
This keeps the notebook and the CLI scripts in sync.

In [ ]:
import os
os.makedirs("src", exist_ok=True)

In [ ]:
%%writefile src/utils.py
"""Shared helpers: reproducibility, device selection, checkpoint I/O."""
import json
import random
from pathlib import Path

import numpy as np
import torch

CLASS_NAMES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]


def set_seed(seed: int = 42) -> None:
    """Fix all relevant RNGs so runs (splits, init, augmentation) are reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def save_checkpoint(model: torch.nn.Module, path: str, extra: dict | None = None) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    payload = {"model_state": model.state_dict()}
    if extra:
        payload.update(extra)
    torch.save(payload, path)


def load_checkpoint(model: torch.nn.Module, path: str, map_location=None) -> dict:
    payload = torch.load(path, map_location=map_location or get_device())
    model.load_state_dict(payload["model_state"])
    return payload


def save_json(obj: dict, path: str) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def load_json(path: str) -> dict:
    with open(path) as f:
        return json.load(f)


In [ ]:
%%writefile src/transforms.py
"""
Preprocessing + augmentation pipelines, matching Section 3(2) of the paper.

- All images: resize to 224x224 (ImageNet-pretrained ResNet input size),
  cast to tensor, normalize with ImageNet channel mean/std.
- "heavy" augmentation: random rotation, zoom (via RandomResizedCrop),
  horizontal flip, brightness jitter -- applied to train split only.
- "none": just the deterministic resize/normalize, used as the ablation
  control and always used for val/test regardless of the train setting.
"""
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224


def eval_transform() -> transforms.Compose:
    """Deterministic preprocessing used for val/test, and for train in the
    'no augmentation' ablation arm."""
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def heavy_train_transform() -> transforms.Compose:
    """Heavy augmentation arm: rotation, zoom/crop, horizontal flip,
    brightness jitter, then the same normalization as eval."""
    return transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),  # zoom
        transforms.RandomRotation(degrees=20),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.3),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def get_train_transform(augment: bool) -> transforms.Compose:
    return heavy_train_transform() if augment else eval_transform()


In [ ]:
%%writefile src/dataset.py
"""TrashNet Dataset + DataLoader construction from the CSV splits."""
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from utils import CLASS_NAMES

LABEL_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}


class TrashNetDataset(Dataset):
    def __init__(self, csv_path: str, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = LABEL_TO_IDX[row["label"]]
        return image, torch.tensor(label, dtype=torch.long)


def build_dataloader(
    csv_path: str,
    transform,
    batch_size: int,
    shuffle: bool,
    num_workers: int = 2,
) -> DataLoader:
    ds = TrashNetDataset(csv_path, transform=transform)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )


def class_counts(csv_path: str) -> pd.Series:
    """Useful for sanity-checking stratification and for optional class weighting."""
    df = pd.read_csv(csv_path)
    return df["label"].value_counts()


In [ ]:
%%writefile src/model.py
"""
ResNet50 backbone + custom classification head (Section 3, steps 3-4),
with support for staged/progressive unfreezing:

  Stage 0: only the new head is trainable (backbone fully frozen).
  Stage 1: unfreeze layer4 (the last residual block) at a lower LR.
  Stage 2: unfreeze layer3 + layer4.
  Stage 3: unfreeze everything (full fine-tune).

This lets train.py implement "train the head first, then progressively
unfreeze deeper layers at a lower learning rate" instead of unfreezing
everything at once.
"""
import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import ResNet50_Weights

from utils import CLASS_NAMES

NUM_CLASSES = len(CLASS_NAMES)

# ResNet50's children in unfreezing order, deepest-first, matching the
# stages described above. layer4 is unfrozen before layer3, etc.
UNFREEZE_STAGES = {
    0: [],                                  # head only
    1: ["layer4"],
    2: ["layer4", "layer3"],
    3: ["layer4", "layer3", "layer2", "layer1", "conv1", "bn1"],  # full fine-tune
}


class ResNetWasteClassifier(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, dropout: float = 0.3):
        super().__init__()
        backbone = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        in_features = backbone.fc.in_features

        # Drop the original ImageNet fc layer; keep everything up to
        # global average pooling (backbone.avgpool is already GAP).
        backbone.fc = nn.Identity()
        self.backbone = backbone

        # Classification head: GAP (already applied inside backbone) ->
        # Dropout -> Dense -> ReLU -> Dropout -> Dense (logits).
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(256, num_classes),
        )

        # Start fully frozen; train.py calls set_unfreeze_stage() to
        # progressively open up layers.
        self.freeze_backbone()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)  # (B, in_features), GAP already applied
        return self.head(features)  # (B, num_classes) logits (softmax via loss fn)

    def freeze_backbone(self) -> None:
        for p in self.backbone.parameters():
            p.requires_grad = False

    def set_unfreeze_stage(self, stage: int) -> None:
        """Freeze everything, then unfreeze only the named submodules for
        this stage. Stage 0 = head-only training."""
        assert stage in UNFREEZE_STAGES, f"stage must be one of {list(UNFREEZE_STAGES)}"
        self.freeze_backbone()
        for name in UNFREEZE_STAGES[stage]:
            module = getattr(self.backbone, name)
            for p in module.parameters():
                p.requires_grad = True

    def trainable_parameters(self):
        return [p for p in self.parameters() if p.requires_grad]


def build_model(dropout: float = 0.3, device=None) -> ResNetWasteClassifier:
    model = ResNetWasteClassifier(dropout=dropout)
    if device is not None:
        model = model.to(device)
    return model


In [ ]:
%%writefile src/train.py
"""
Core training loop implementing the staged/progressive unfreezing schedule
from Section 3(3): train the head first, then progressively unfreeze
deeper layers at a lower learning rate.

Used directly for a one-off run, and imported by tune.py / ablation.py so
the same training logic backs the hyperparameter search and the
augmentation ablation.
"""
import argparse
import copy
import time

import torch
import torch.nn as nn
from torch.optim import SGD, Adam

from dataset import build_dataloader
from model import build_model
from transforms import eval_transform, get_train_transform
from utils import get_device, save_checkpoint, set_seed

# Staged schedule: (unfreeze_stage, epochs, lr_multiplier)
# lr_multiplier scales down the base LR as we open up more of the backbone,
# so already-good ImageNet features aren't clobbered by large gradient steps.
DEFAULT_SCHEDULE = [
    (0, 5, 1.0),   # head only
    (1, 5, 0.5),   # + layer4
    (2, 5, 0.25),  # + layer3
    (3, 5, 0.1),   # full fine-tune
]


def build_optimizer(name: str, params, lr: float):
    if name == "adam":
        return Adam(params, lr=lr)
    if name == "sgd_momentum":
        return SGD(params, lr=lr, momentum=0.9)
    raise ValueError(f"Unknown optimizer: {name}")


@torch.no_grad()
def evaluate_loss_acc(model, loader, criterion, device):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        loss_sum += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)
    return loss_sum / total, correct / total


def train_model(
    train_csv: str,
    val_csv: str,
    lr: float = 1e-4,
    optimizer_name: str = "adam",
    batch_size: int = 32,
    dropout: float = 0.3,
    augment: bool = True,
    schedule=None,
    patience: int = 5,
    seed: int = 42,
    checkpoint_path: str | None = None,
    verbose: bool = True,
):
    """Runs the full staged fine-tuning schedule. Returns the best model
    (by validation accuracy, ties by lower validation loss) and a history
    dict, so callers (Optuna, ablation) can inspect val performance without
    ever touching the test split."""
    set_seed(seed)
    device = get_device()
    schedule = schedule or DEFAULT_SCHEDULE

    train_loader = build_dataloader(
        train_csv, get_train_transform(augment), batch_size, shuffle=True
    )
    val_loader = build_dataloader(val_csv, eval_transform(), batch_size, shuffle=False)

    model = build_model(dropout=dropout, device=device)
    criterion = nn.CrossEntropyLoss()

    best_val_acc, best_val_loss = -1.0, float("inf")
    best_state = None
    epochs_no_improve = 0
    history = []

    for stage, n_epochs, lr_mult in schedule:
        model.set_unfreeze_stage(stage)
        optimizer = build_optimizer(
            optimizer_name, model.trainable_parameters(), lr * lr_mult
        )

        for epoch in range(n_epochs):
            model.train()
            running_loss, running_correct, seen = 0.0, 0, 0
            t0 = time.time()
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                logits = model(images)
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * images.size(0)
                running_correct += (logits.argmax(dim=1) == labels).sum().item()
                seen += images.size(0)

            train_loss = running_loss / seen
            train_acc = running_correct / seen
            val_loss, val_acc = evaluate_loss_acc(model, val_loader, criterion, device)

            history.append({
                "stage": stage, "train_loss": train_loss, "train_acc": train_acc,
                "val_loss": val_loss, "val_acc": val_acc,
            })
            if verbose:
                print(
                    f"[stage {stage}] epoch {epoch+1}/{n_epochs} "
                    f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
                    f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} "
                    f"({time.time()-t0:.1f}s)"
                )

            # Early stopping tracks val_acc first, val_loss as tiebreaker/overfit check.
            improved = val_acc > best_val_acc or (
                val_acc == best_val_acc and val_loss < best_val_loss
            )
            if improved:
                best_val_acc, best_val_loss = val_acc, val_loss
                best_state = copy.deepcopy(model.state_dict())
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    if verbose:
                        print(f"Early stopping (no improvement for {patience} epochs).")
                    break

    model.load_state_dict(best_state)
    if checkpoint_path:
        save_checkpoint(model, checkpoint_path, extra={
            "val_acc": best_val_acc, "val_loss": best_val_loss,
            "hparams": {
                "lr": lr, "optimizer": optimizer_name, "batch_size": batch_size,
                "dropout": dropout, "augment": augment,
            },
        })
    return model, {"best_val_acc": best_val_acc, "best_val_loss": best_val_loss, "history": history}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--splits_dir", default="data/splits")
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--optimizer", default="adam", choices=["adam", "sgd_momentum"])
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--dropout", type=float, default=0.3)
    ap.add_argument("--no_augment", action="store_true")
    ap.add_argument("--checkpoint", default="checkpoints/best_model.pt")
    args = ap.parse_args()

    _, result = train_model(
        train_csv=f"{args.splits_dir}/train.csv",
        val_csv=f"{args.splits_dir}/val.csv",
        lr=args.lr,
        optimizer_name=args.optimizer,
        batch_size=args.batch_size,
        dropout=args.dropout,
        augment=not args.no_augment,
        checkpoint_path=args.checkpoint,
    )
    print(f"\nBest val_acc={result['best_val_acc']:.4f}, val_loss={result['best_val_loss']:.4f}")
    print(f"Checkpoint saved to {args.checkpoint}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/make_splits.py
"""
Build stratified 70/15/15 train/val/test splits from a TrashNet folder tree.

Expected input layout:
    raw_dir/
        cardboard/*.jpg
        glass/*.jpg
        metal/*.jpg
        paper/*.jpg
        plastic/*.jpg
        trash/*.jpg

Writes train.csv / val.csv / test.csv (columns: filepath,label) into out_dir.
This is run ONCE. The test split is not touched again until src/evaluate.py,
so validation-set decisions (hyperparameter tuning, ablation) never leak
into the reported test performance -- per Section 4.1 of the paper.
"""
import argparse
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

from utils import CLASS_NAMES, set_seed

VALID_EXT = {".jpg", ".jpeg", ".png"}


def collect_files(raw_dir: Path) -> pd.DataFrame:
    rows = []
    for cls in CLASS_NAMES:
        cls_dir = raw_dir / cls
        if not cls_dir.exists():
            raise FileNotFoundError(
                f"Expected class folder '{cls_dir}' not found. "
                f"Check that --raw_dir points at the unzipped TrashNet root."
            )
        for p in cls_dir.iterdir():
            if p.suffix.lower() in VALID_EXT:
                rows.append({"filepath": str(p.resolve()), "label": cls})
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No images found -- check --raw_dir path and folder names.")
    return df


def make_splits(df: pd.DataFrame, seed: int):
    # First peel off 70% train, 30% temp; then split temp 50/50 -> 15%/15%.
    train_df, temp_df = train_test_split(
        df, test_size=0.30, stratify=df["label"], random_state=seed
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df["label"], random_state=seed
    )
    return train_df, val_df, test_df


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw_dir", required=True, help="Path to unzipped TrashNet root")
    ap.add_argument("--out_dir", default="data/splits")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    set_seed(args.seed)
    df = collect_files(Path(args.raw_dir))

    print("Class distribution (full dataset):")
    print(df["label"].value_counts(), "\n")

    train_df, val_df, test_df = make_splits(df, args.seed)

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(out_dir / "train.csv", index=False)
    val_df.to_csv(out_dir / "val.csv", index=False)
    test_df.to_csv(out_dir / "test.csv", index=False)

    for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
        print(f"{name}: {len(split)} images")
        print(split["label"].value_counts(normalize=True).round(3), "\n")

    print(f"Splits written to {out_dir}/. The test.csv split should not be "
          f"opened again until final evaluation.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/tune.py
"""
Section 4.2 - Hyperparameter Tuning.

Uses Optuna to search:
  - Learning Rate:  {1e-3, 1e-4, 1e-5}
  - Optimizer:      {adam, sgd_momentum}
  - Batch Size:     {16, 32, 64}
  - Dropout Rate:   {0.2, 0.3, 0.5}

Each trial is scored on validation accuracy (val loss used as a tiebreaker
and to flag overfitting -- see train.py's early-stopping logic, which
already tracks both). The winning config is written to best_params.json;
the held-out test split is never touched here.
"""
import argparse

import optuna

from train import train_model
from utils import save_json


def make_objective(splits_dir: str, augment: bool, patience: int):
    train_csv = f"{splits_dir}/train.csv"
    val_csv = f"{splits_dir}/val.csv"

    def objective(trial: optuna.Trial) -> float:
        lr = trial.suggest_categorical("lr", [1e-3, 1e-4, 1e-5])
        optimizer_name = trial.suggest_categorical("optimizer", ["adam", "sgd_momentum"])
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
        dropout = trial.suggest_categorical("dropout", [0.2, 0.3, 0.5])

        _, result = train_model(
            train_csv=train_csv,
            val_csv=val_csv,
            lr=lr,
            optimizer_name=optimizer_name,
            batch_size=batch_size,
            dropout=dropout,
            augment=augment,
            patience=patience,
            verbose=False,
        )

        # Stash val_loss on the trial so we can use it as a tiebreaker /
        # overfitting check when comparing trials with tied accuracy.
        trial.set_user_attr("val_loss", result["best_val_loss"])
        return result["best_val_acc"]

    return objective


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--splits_dir", default="data/splits")
    ap.add_argument("--n_trials", type=int, default=30)
    ap.add_argument("--patience", type=int, default=3,
                     help="Shorter patience during search to keep trials cheap.")
    ap.add_argument("--augment", action="store_true", default=True,
                     help="Use heavy augmentation during the search "
                          "(the augmentation ablation itself happens separately).")
    ap.add_argument("--out", default="outputs/best_params.json")
    args = ap.parse_args()

    study = optuna.create_study(direction="maximize", study_name="waste_resnet50_tuning")
    study.optimize(
        make_objective(args.splits_dir, args.augment, args.patience),
        n_trials=args.n_trials,
    )

    print("Best trial:")
    print(f"  val_acc:  {study.best_value:.4f}")
    print(f"  val_loss: {study.best_trial.user_attrs.get('val_loss'):.4f}")
    print(f"  params:   {study.best_params}")

    save_json({
        "best_params": study.best_params,
        "best_val_acc": study.best_value,
        "best_val_loss": study.best_trial.user_attrs.get("val_loss"),
    }, args.out)
    print(f"\nSaved to {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/ablation.py
"""
Section 4.3 - Data Augmentation Ablation.

Using the best hyperparameter configuration from tune.py, trains two
otherwise-identical models -- one with heavy augmentation (rotation, zoom,
flip, brightness), one with none -- and evaluates BOTH on the same
held-out test split, to see whether augmentation actually helps
generalization on this small dataset.
"""
import argparse

from evaluate import full_evaluate
from train import train_model
from utils import load_json, save_json


def run_ablation(splits_dir: str, best_params: dict, out_dir: str = "outputs"):
    train_csv = f"{splits_dir}/train.csv"
    val_csv = f"{splits_dir}/val.csv"
    test_csv = f"{splits_dir}/test.csv"

    results = {}
    for augment in (True, False):
        arm = "heavy_augmentation" if augment else "no_augmentation"
        print(f"\n=== Training arm: {arm} ===")
        model, train_result = train_model(
            train_csv=train_csv,
            val_csv=val_csv,
            lr=best_params["lr"],
            optimizer_name=best_params["optimizer"],
            batch_size=best_params["batch_size"],
            dropout=best_params["dropout"],
            augment=augment,
            checkpoint_path=f"checkpoints/ablation_{arm}.pt",
        )
        print(f"[{arm}] best val_acc={train_result['best_val_acc']:.4f}")

        test_metrics = full_evaluate(
            model, test_csv, batch_size=best_params["batch_size"],
            save_confusion_matrix_path=f"{out_dir}/confusion_matrix_{arm}.png",
        )
        results[arm] = {
            "val_acc": train_result["best_val_acc"],
            "val_loss": train_result["best_val_loss"],
            "test_accuracy": test_metrics["accuracy"],
            "test_macro_precision": test_metrics["macro_precision"],
            "test_macro_recall": test_metrics["macro_recall"],
            "test_macro_f1": test_metrics["macro_f1"],
            "per_class": test_metrics["per_class"],
        }
        print(f"[{arm}] test_accuracy={test_metrics['accuracy']:.4f} "
              f"macro_f1={test_metrics['macro_f1']:.4f}")

    save_json(results, f"{out_dir}/ablation_results.json")
    print(f"\nAblation results saved to {out_dir}/ablation_results.json")
    print(f"Confusion matrices saved to {out_dir}/confusion_matrix_<arm>.png")

    diff = (results["heavy_augmentation"]["test_accuracy"]
            - results["no_augmentation"]["test_accuracy"])
    print(f"\nAugmentation effect on test accuracy: {diff:+.4f} "
          f"({'helped' if diff > 0 else 'hurt' if diff < 0 else 'no change'})")
    return results


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--splits_dir", default="data/splits")
    ap.add_argument("--best_params", default="outputs/best_params.json",
                     help="Path to best_params.json produced by tune.py")
    ap.add_argument("--out_dir", default="outputs")
    args = ap.parse_args()

    params_blob = load_json(args.best_params)
    best_params = params_blob["best_params"]

    run_ablation(args.splits_dir, best_params, args.out_dir)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/evaluate.py
"""
Section 4.4 - Evaluation Metrics.

Reports, on the held-out test split only:
  - Overall accuracy
  - Per-class and macro-averaged precision / recall / F1
  - Confusion matrix (saved as PNG), so misclassification patterns
    like "glass mistaken for clear plastic" are visible directly.
"""
import argparse

import matplotlib

matplotlib.use("Agg")  # headless-safe (Colab/servers)
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_fscore_support,
)

from dataset import build_dataloader
from model import build_model
from transforms import eval_transform
from utils import CLASS_NAMES, get_device, load_checkpoint


@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)


def plot_confusion_matrix(cm: np.ndarray, class_names: list, save_path: str) -> None:
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title("Confusion Matrix (Test Split)")

    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], "d"), ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")

    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)


def full_evaluate(
    model,
    test_csv: str,
    batch_size: int = 32,
    save_confusion_matrix_path: str = "outputs/confusion_matrix.png",
) -> dict:
    device = get_device()
    model = model.to(device)
    test_loader = build_dataloader(test_csv, eval_transform(), batch_size, shuffle=False)

    y_true, y_pred = collect_predictions(model, test_loader, device)

    accuracy = float((y_true == y_pred).mean())
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(CLASS_NAMES)), zero_division=0
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    per_class = {
        CLASS_NAMES[i]: {
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "support": int(support[i]),
        }
        for i in range(len(CLASS_NAMES))
    }

    cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)))
    plot_confusion_matrix(cm, CLASS_NAMES, save_confusion_matrix_path)

    return {
        "accuracy": accuracy,
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "per_class": per_class,
        "confusion_matrix": cm.tolist(),
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--checkpoint", required=True)
    ap.add_argument("--splits_dir", default="data/splits")
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--dropout", type=float, default=0.3,
                     help="Must match the dropout the checkpoint was trained with.")
    ap.add_argument("--out", default="outputs/test_metrics.png")
    args = ap.parse_args()

    device = get_device()
    model = build_model(dropout=args.dropout, device=device)
    payload = load_checkpoint(model, args.checkpoint, map_location=device)
    if "hparams" in payload:
        print(f"Loaded checkpoint trained with: {payload['hparams']}")

    metrics = full_evaluate(
        model, f"{args.splits_dir}/test.csv",
        batch_size=args.batch_size, save_confusion_matrix_path=args.out,
    )

    print(f"\nTest Accuracy: {metrics['accuracy']:.4f}")
    print(f"Macro Precision: {metrics['macro_precision']:.4f}")
    print(f"Macro Recall:    {metrics['macro_recall']:.4f}")
    print(f"Macro F1:        {metrics['macro_f1']:.4f}\n")
    print("Per-class breakdown:")
    for cls, m in metrics["per_class"].items():
        print(f"  {cls:10s} precision={m['precision']:.3f} recall={m['recall']:.3f} "
              f"f1={m['f1']:.3f} support={m['support']}")
    print(f"\nConfusion matrix saved to {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
import sys
sys.path.insert(0, "src")
os.makedirs("outputs", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)

## 1. Stratified 70/15/15 split (Section 4.1)

Runs once. Writes `data/splits/{train,val,test}.csv`. The test split is not
opened again until the final evaluation cell at the bottom of this notebook.

In [ ]:
from make_splits import collect_files, make_splits
from pathlib import Path
from utils import set_seed

set_seed(42)
df = collect_files(Path("data/raw"))
print("Class distribution (full dataset):")
print(df["label"].value_counts())

train_df, val_df, test_df = make_splits(df, seed=42)

out_dir = Path("data/splits")
out_dir.mkdir(parents=True, exist_ok=True)
train_df.to_csv(out_dir / "train.csv", index=False)
val_df.to_csv(out_dir / "val.csv", index=False)
test_df.to_csv(out_dir / "test.csv", index=False)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{name}: {len(split)} images")
    print(split["label"].value_counts(normalize=True).round(3))

## 2. Baseline training run (Section 3)

A single run of the full pipeline: ResNet50 backbone, ImageNet normalization,
heavy augmentation, and the staged/progressive unfreezing schedule
(head-only -> +layer4 -> +layer3 -> full fine-tune). This is mainly a sanity
check before the more expensive hyperparameter search below.

In [ ]:
from train import train_model

model, result = train_model(
    train_csv="data/splits/train.csv",
    val_csv="data/splits/val.csv",
    lr=1e-4,
    optimizer_name="adam",
    batch_size=32,
    dropout=0.3,
    augment=True,
    checkpoint_path="checkpoints/baseline_model.pt",
)
print(f"\nBaseline best val_acc={result['best_val_acc']:.4f}, val_loss={result['best_val_loss']:.4f}")

## 3. Hyperparameter search (Section 4.2)

Optuna searches learning rate, optimizer, batch size, and dropout. Each trial
is scored on validation accuracy (val loss recorded as a tiebreaker / overfit
check). Adjust `n_trials` down if you're short on GPU time -- 30 trials at
~20 epochs each on ~1,769 training images comfortably fits a T4 in a few hours,
but you can start with 10-15 to sanity check first.

In [ ]:
from tune import make_objective
import optuna

N_TRIALS = 15  # raise to 30 for the full search described in the paper

study = optuna.create_study(direction="maximize", study_name="waste_resnet50_tuning")
study.optimize(
    make_objective(splits_dir="data/splits", augment=True, patience=3),
    n_trials=N_TRIALS,
)

print("Best trial:")
print(f"  val_acc:  {study.best_value:.4f}")
print(f"  val_loss: {study.best_trial.user_attrs.get('val_loss'):.4f}")
print(f"  params:   {study.best_params}")

In [ ]:
from utils import save_json

best_params_blob = {
    "best_params": study.best_params,
    "best_val_acc": study.best_value,
    "best_val_loss": study.best_trial.user_attrs.get("val_loss"),
}
save_json(best_params_blob, "outputs/best_params.json")
print("Saved outputs/best_params.json")

## 4. Data augmentation ablation (Section 4.3)

Using the winning hyperparameters from the search above, trains one model
with heavy augmentation and one with none, then evaluates both on the same
held-out test split.

In [ ]:
from ablation import run_ablation
from utils import load_json

best_params = load_json("outputs/best_params.json")["best_params"]
ablation_results = run_ablation("data/splits", best_params, out_dir="outputs")

In [ ]:
import json
print(json.dumps(ablation_results, indent=2))

## 5. Final evaluation on the held-out test split (Section 4.4)

Loads the best-performing checkpoint (whichever ablation arm scored higher
on the test set above) and reports overall accuracy, per-class and
macro-averaged precision/recall/F1, and a confusion matrix.

In [ ]:
from model import build_model
from evaluate import full_evaluate
from utils import get_device, load_checkpoint

best_arm = "heavy_augmentation" if (
    ablation_results["heavy_augmentation"]["test_accuracy"]
    >= ablation_results["no_augmentation"]["test_accuracy"]
) else "no_augmentation"
print("Best-performing arm on test set:", best_arm)

device = get_device()
final_model = build_model(dropout=best_params["dropout"], device=device)
load_checkpoint(final_model, f"checkpoints/ablation_{best_arm}.pt", map_location=device)

metrics = full_evaluate(
    final_model, "data/splits/test.csv",
    batch_size=best_params["batch_size"],
    save_confusion_matrix_path="outputs/final_confusion_matrix.png",
)

print(f"\nFinal Test Accuracy: {metrics['accuracy']:.4f}")
print(f"Macro Precision: {metrics['macro_precision']:.4f}")
print(f"Macro Recall:    {metrics['macro_recall']:.4f}")
print(f"Macro F1:        {metrics['macro_f1']:.4f}\n")
print("Per-class breakdown:")
for cls, m in metrics["per_class"].items():
    print(f"  {cls:10s} precision={m['precision']:.3f} recall={m['recall']:.3f} "
          f"f1={m['f1']:.3f} support={m['support']}")

In [ ]:
from IPython.display import Image, display
display(Image("outputs/final_confusion_matrix.png"))